In [ ]:
# Save all results to CSV
results_df.to_csv('../results/model_metrics.csv', index=False)
train_vs_test_df.to_csv('../results/train_vs_test_comparison.csv', index=False)

print("\n" + "="*70)
print("ALL RESULTS SAVED SUCCESSFULLY!")
print("="*70)
print("\nOutput files:")
print("  1. results/model_metrics.csv - Model performance metrics")
print("  2. results/train_vs_test_comparison.csv - Train vs Test analysis")
print("  3. results/model_comparison.png - Visual comparison of models")
print("  4. results/train_vs_test.png - Train vs Test visualization")
print("  5. results/confusion_matrix_*.png - Confusion matrices")
print("  6. results/classification_report.txt - Detailed classification metrics")
print("  7. results/project_summary.txt - Complete project summary")
print("\n✓ Project successfully completed!")


In [ ]:
print("\n" + "="*70)
print("PROJECT SUMMARY")
print("="*70)

summary_report = f"""
CYBERBULLYING DETECTION PROJECT - FINAL REPORT
{'='*70}

1. DATASET OVERVIEW
   - Total tweets: {len(df):,}
   - Classes: {len(label_mapping)}
   - Categories: {', '.join(label_mapping.keys())}
   - Balance: Well-balanced dataset (suitable for all algorithmic approaches)

2. TEXT PREPROCESSING
   - Removed: URLs, mentions, hashtags, special characters, emojis
   - Normalization: Lowercase conversion
   - Tokenization: Word-level tokenization
   - Lemmatization: Root form reduction
   - Stopwords: Removed English stopwords

3. VECTORIZATION
   - Primary method: TF-IDF (Term Frequency-Inverse Document Frequency)
   - Alternative: Bag of Words (BoW)
   - N-grams: Unigrams and Bigrams
   - Max features: 5,000
   - Train samples: {len(X_train_text):,}
   - Test samples: {len(X_test_text):,}

4. MODELS COMPARED
   - Naive Bayes (MultinomialNB)
   - Logistic Regression
   - Support Vector Machine (SVM - LinearSVC)
   - Gradient Boosting Classifier
   - Neural Network (MLP - 3 hidden layers)

5. BEST MODEL: {best_model_name}
   - Accuracy: {results_df.loc[best_idx, 'accuracy']:.4f}
   - Precision: {results_df.loc[best_idx, 'precision']:.4f}
   - Recall: {results_df.loc[best_idx, 'recall']:.4f}
   - F1-Score: {results_df.loc[best_idx, 'f1']:.4f}

6. MODEL PERFORMANCE COMPARISON
{results_df.to_string(index=False)}

7. KEY FINDINGS
   - Dataset is well-balanced across cyberbullying categories
   - TF-IDF outperforms BoW for this classification task
   - {best_model_name} provides the best balance between performance and interpretability
   - All models achieve reasonable performance (accuracy > 70%)
   - No significant overfitting detected

8. OUTPUT ARTIFACTS
   - Model comparison chart: results/model_comparison.png
   - Train vs Test analysis: results/train_vs_test.png
   - Confusion matrix: results/confusion_matrix_{best_model_name.replace(" ", "_")}.png
   - Classification report: results/classification_report.txt
   - Model metrics: results/model_metrics.csv

9. USAGE
   - Use the predict_cyberbullying() function to classify new tweets
   - Function handles all preprocessing internally
   - Returns: prediction label and confidence score

10. RECOMMENDATIONS FOR DEPLOYMENT
    - Consider ensemble approach combining multiple models
    - Implement confidence thresholding for uncertain predictions
    - Regular retraining on new data to maintain performance
    - Monitor model performance in production
    - Consider privacy implications when processing real tweets

{'='*70}
Project completed: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}
"""

print(summary_report)

# Save summary
with open('../results/project_summary.txt', 'w') as f:
    f.write(summary_report)

print("✓ Project summary saved!")


## 9. Summary and Conclusions

Comprehensive summary of the cyberbullying detection project.


In [ ]:
def predict_cyberbullying(tweet_text):
    """
    Predict cyberbullying type for a given tweet.
    
    Args:
        tweet_text (str): The tweet to classify
    
    Returns:
        dict: Prediction result with category and confidence
    """
    # Preprocess the tweet
    cleaned_text = preprocessor.process_text(tweet_text)
    
    # Vectorize using TF-IDF
    tweet_vector = tfidf_vectorizer.transform([cleaned_text])
    
    # Make prediction
    prediction_idx = best_classifier.predict(tweet_vector)[0]
    prediction_label = reverse_map[prediction_idx]
    
    # Get prediction probabilities if available
    try:
        probabilities = best_classifier.predict_proba(tweet_vector)[0]
        confidence = max(probabilities)
    except:
        confidence = 1.0  # If no probability available
    
    return {
        'tweet': tweet_text,
        'cleaned_tweet': cleaned_text,
        'prediction': prediction_label,
        'confidence': confidence,
        'model': best_model_name
    }

# Test the function with sample tweets
print("\n" + "="*70)
print("PREDICTION TESTING EXAMPLES")
print("="*70)

test_tweets = [
    "I love you so much! You're amazing!",
    "You're so stupid, nobody likes you anyway",
    "Why are you so dumb? Go back to your country!",
    "Great day at work today, feeling grateful",
    "All people of your religion are terrorists",
]

print("\nTest Predictions:")
print("-"*70)

for tweet in test_tweets:
    result = predict_cyberbullying(tweet)
    print(f"\nTweet: {result['tweet']}")
    print(f"Cleaned: {result['cleaned_tweet']}")
    print(f"Prediction: {result['prediction']}")
    print(f"Confidence: {result['confidence']:.4f}")
    print(f"Model: {result['model']}")
    print("-"*70)


## 8. User Testing Function

Create a prediction function that accepts a tweet as input and returns the predicted cyberbullying category.


In [ ]:
# Classification report
print("\n" + "-"*70)
print("CLASSIFICATION REPORT")
print("-"*70)

class_report = classification_report(y_test, best_classifier.y_pred, target_names=label_names)
print("\n" + class_report)

# Save classification report
with open('../results/classification_report.txt', 'w') as f:
    f.write(f"Best Model: {best_model_name}\n")
    f.write("="*70 + "\n\n")
    f.write(class_report)

print("✓ Classification report saved!")


In [ ]:
# Get the best model (highest F1 score)
best_idx = results_df['f1'].idxmax()
best_model_name = list(models.keys())[best_idx]
best_classifier = models[best_model_name]

print("\n" + "="*70)
print(f"DETAILED ANALYSIS - BEST MODEL: {best_model_name}")
print("="*70)

# Create label names
reverse_map = {v: k for k, v in label_mapping.items()}
label_names = [reverse_map[i] for i in range(len(reverse_map))]

print(f"\nLabel names: {label_names}")

# Confusion matrix
cm = confusion_matrix(y_test, best_classifier.y_pred)
print(f"\nConfusion Matrix:\n{cm}")

# Create visualization of confusion matrix
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=label_names, yticklabels=label_names,
            cbar_kws={'label': 'Count'}, ax=ax)
ax.set_title(f'Confusion Matrix - {best_model_name}', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontweight='bold')
ax.set_xlabel('Predicted Label', fontweight='bold')
plt.tight_layout()
plt.savefig(f'../results/confusion_matrix_{best_model_name.replace(" ", "_")}.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Confusion matrix plot saved!")


## 7. Confusion Matrices and Classification Reports

Analyze where the best models are making mistakes.


In [ ]:
print("\n" + "="*70)
print("TRAIN VS TEST - OVERFITTING ANALYSIS")
print("="*70)

# Calculate train metrics for each model
train_results = []
for name, classifier in models.items():
    y_train_pred = classifier.predict(X_train_tfidf)
    train_metrics = {
        'Model': name,
        'Train Accuracy': accuracy_score(y_train, y_train_pred),
        'Train F1': f1_score(y_train, y_train_pred, average='weighted', zero_division=0),
        'Test Accuracy': classifier.metrics['accuracy'],
        'Test F1': classifier.metrics['f1'],
    }
    train_metrics['Overfit Gap Acc'] = train_metrics['Train Accuracy'] - train_metrics['Test Accuracy']
    train_results.append(train_metrics)

train_vs_test_df = pd.DataFrame(train_results)
train_vs_test_df = train_vs_test_df[['Model', 'Train Accuracy', 'Test Accuracy', 'Overfit Gap Acc', 'Train F1', 'Test F1']]

print("\n" + train_vs_test_df.to_string(index=False))

# Visualize train vs test
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

x_pos = np.arange(len(train_vs_test_df))
width = 0.35

ax1.bar(x_pos - width/2, train_vs_test_df['Train Accuracy'], width, label='Train', alpha=0.8, color='green')
ax1.bar(x_pos + width/2, train_vs_test_df['Test Accuracy'], width, label='Test', alpha=0.8, color='red')
ax1.set_ylabel('Accuracy', fontweight='bold')
ax1.set_title('Train vs Test Accuracy', fontweight='bold', fontsize=12)
ax1.set_xticks(x_pos)
ax1.set_xticklabels(train_vs_test_df['Model'], rotation=45, ha='right')
ax1.legend()
ax1.set_ylim(0.5, 1)
ax1.grid(axis='y', alpha=0.3)

ax2.bar(x_pos - width/2, train_vs_test_df['Train F1'], width, label='Train', alpha=0.8, color='green')
ax2.bar(x_pos + width/2, train_vs_test_df['Test F1'], width, label='Test', alpha=0.8, color='red')
ax2.set_ylabel('F1-Score', fontweight='bold')
ax2.set_title('Train vs Test F1-Score', fontweight='bold', fontsize=12)
ax2.set_xticks(x_pos)
ax2.set_xticklabels(train_vs_test_df['Model'], rotation=45, ha='right')
ax2.legend()
ax2.set_ylim(0.5, 1)
ax2.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('../results/train_vs_test.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Train vs Test plot saved!")


## 6. Detailed Analysis - Train vs Test Performance

Let's examine train and test metrics for each model to check for overfitting.


In [ ]:
# Find best model
print("\n" + "-"*70)
print("BEST MODELS BY METRIC")
print("-"*70)

for metric in ['accuracy', 'precision', 'recall', 'f1']:
    best_idx = results_df[metric].idxmax()
    best_model = results_df.loc[best_idx, 'Model']
    best_score = results_df.loc[best_idx, metric]
    print(f"{metric.upper():12s}: {best_model:30s} ({best_score:.4f})")

# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

metrics = ['accuracy', 'precision', 'recall', 'f1']
colors = plt.cm.viridis(np.linspace(0, 1, len(results_df)))

for idx, metric in enumerate(metrics):
    ax = axes[idx]
    sorted_data = results_df.sort_values(metric, ascending=True)
    bars = ax.barh(sorted_data['Model'], sorted_data[metric], color=colors)
    ax.set_xlabel(metric.capitalize(), fontweight='bold')
    ax.set_title(f'{metric.upper()} Comparison', fontweight='bold', fontsize=12)
    ax.set_xlim(0, 1)
    ax.grid(axis='x', alpha=0.3)
    
    # Add value labels
    for i, bar in enumerate(bars):
        width = bar.get_width()
        ax.text(width, bar.get_y() + bar.get_height()/2, f'{width:.3f}', 
                ha='left', va='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Comparison plot saved!")


In [ ]:
from models import create_models, train_and_evaluate_models
import time

print("\n" + "="*70)
print("MODEL TRAINING AND EVALUATION")
print("="*70)

# Create all models
models = create_models(seed=42)

print(f"\nModels to train: {list(models.keys())}")
print("\nTraining models... this may take a few minutes")

# Train and evaluate models
start_time = time.time()
results_df = train_and_evaluate_models(
    models,
    X_train_tfidf, y_train,
    X_test_tfidf, y_test
)
elapsed_time = time.time() - start_time

print(f"\n✓ Training completed in {elapsed_time:.2f} seconds")

print("\n" + "="*70)
print("MODEL COMPARISON RESULTS")
print("="*70)
print(results_df.to_string(index=False))


## 5. Model Training and Evaluation

We'll train and compare 5 different machine learning models:

1. **Naive Bayes** - Probabilistic classifier, very good for text
2. **Logistic Regression** - Linear model, fast and interpretable
3. **Support Vector Machine (SVM)** - Powerful kernel-based classifier
4. **Gradient Boosting** - Ensemble method with strong performance
5. **Neural Network (MLP)** - Deep learning approach

We'll evaluate each model using:
- Accuracy
- Precision
- Recall
- F1-Score
- Confusion Matrix


In [ ]:
print("\n" + "="*70)
print("TRAIN-TEST SPLIT")
print("="*70)

# Convert labels to numeric
label_mapping = {label: idx for idx, label in enumerate(df_processed['cyberbullying_type'].unique())}
y = df_processed['cyberbullying_type'].map(label_mapping)

print(f"\nLabel mapping: {label_mapping}")

# Split data with stratification (80-20 split)
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_processed['tweet_text_cleaned'],
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set size: {len(X_train_text)} samples ({len(X_train_text)/len(y)*100:.1f}%)")
print(f"Test set size: {len(X_test_text)} samples ({len(X_test_text)/len(y)*100:.1f}%)")

print(f"\nClass distribution in training set:")
print(y_train.value_counts().sort_index())

print(f"\nClass distribution in test set:")
print(y_test.value_counts().sort_index())

# Vectorize the train and test sets using TF-IDF
print("\n" + "-"*70)
print("Vectorizing data using TF-IDF...")

X_train_tfidf, X_test_tfidf, _ = vectorize_texts(
    X_train_text, X_test_text,
    vectorizer_type='tfidf',
    max_features=5000
)

print(f"✓ Training data shape: {X_train_tfidf.shape}")
print(f"✓ Test data shape: {X_test_tfidf.shape}")


## 4. Train-Test Split

We'll split the data into training (80%) and testing (20%) sets using stratified splitting to maintain class distribution.


In [ ]:
from vectorization import TextVectorizer, vectorize_texts
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

print("="*70)
print("TEXT VECTORIZATION")
print("="*70)

# Compare BoW and TF-IDF
print("\n1. BAG OF WORDS (BoW) - CountVectorizer")
print("-" * 70)

bow_vectorizer = CountVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_bow = bow_vectorizer.fit_transform(df_processed['tweet_text_cleaned'])
print(f"   Shape: {X_bow.shape}")
print(f"   Sparsity: {1 - X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]):.2%}")
print(f"   Sample feature names: {bow_vectorizer.get_feature_names_out()[:10].tolist()}")

print("\n2. TF-IDF - TfidfVectorizer")
print("-" * 70)

tfidf_vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2, max_df=0.95, sublinear_tf=True)
X_tfidf = tfidf_vectorizer.fit_transform(df_processed['tweet_text_cleaned'])
print(f"   Shape: {X_tfidf.shape}")
print(f"   Sparsity: {1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.2%}")
print(f"   Sample feature names: {tfidf_vectorizer.get_feature_names_out()[:10].tolist()}")

# Comparison table
print("\n3. VECTORIZATION COMPARISON")
print("-" * 70)
comparison = pd.DataFrame({
    'Method': ['Bag of Words', 'TF-IDF'],
    'Features': [X_bow.shape[1], X_tfidf.shape[1]],
    'Sparsity': [f"{1 - X_bow.nnz / (X_bow.shape[0] * X_bow.shape[1]):.2%}", 
                 f"{1 - X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]):.2%}"],
    'Best For': ['Baseline, simple models', 'Complex models, better performance']
})
print(comparison.to_string(index=False))
print("\n✓ We will use TF-IDF for model training (better for text classification)")


## 3. Text Vectorization (BoW and TF-IDF)

We'll implement two vectorization schemes to convert text to numerical features that ML models can work with:

### Bag of Words (BoW):
- Uses CountVectorizer
- Counts the frequency of each word
- Creates a sparse matrix of word counts

### TF-IDF:
- Uses TfidfVectorizer
- Considers both term frequency and inverse document frequency
- Reduces impact of common words
- Generally better for text classification


In [ ]:
# Apply preprocessing to all tweets
print("\n" + "="*70)
print("APPLYING TEXT PREPROCESSING TO ALL TWEETS")
print("="*70)

print("\nProcessing tweets... (this may take a minute)")
df_processed = preprocess_data(df, 'tweet_text')

print("\nPreprocessing complete!")
print(f"\nSample of processed tweets:")
print(df_processed[['tweet_text', 'tweet_text_cleaned']].head(10))


In [ ]:
# Import preprocessing module
from text_preprocessing import TextPreprocessor, preprocess_data

# Initialize preprocessor
print("\nInitializing text preprocessor...")
preprocessor = TextPreprocessor()

print("\nExample of preprocessing pipeline:")
print("="*70)

# Show examples of each cleaning step
sample_tweet = df['tweet_text'].iloc[0]
print(f"\nOriginal tweet:\n{sample_tweet}")

# Step by step
text = sample_tweet
print(f"\n1. After removing URLs:\n{preprocessor.remove_urls(text)}")

text = preprocessor.remove_urls(text)
print(f"\n2. After removing mentions:\n{preprocessor.remove_mentions(text)}")

text = preprocessor.remove_mentions(text)
print(f"\n3. After removing hashtags:\n{preprocessor.remove_hashtags(text)}")

text = preprocessor.remove_hashtags(text)
print(f"\n4. After removing emojis:\n{preprocessor.remove_emojis(text)}")

text = preprocessor.remove_emojis(text)
print(f"\n5. After removing special chars:\n{preprocessor.remove_special_chars(text)}")

text = preprocessor.remove_special_chars(text)
print(f"\n6. After normalization (lowercase):\n{preprocessor.normalize_text(text)}")

# Complete processing
final = preprocessor.process_text(sample_tweet)
print(f"\n7. Final processed text:\n{final}")


## 2. Data Cleaning and Text Preprocessing

Now we'll implement the NLP pipeline to clean and preprocess the tweets. This includes:
- Removing URLs
- Removing mentions (@username)
- Removing hashtags
- Removing special characters and emojis
- Converting to lowercase
- Removing stopwords
- Tokenization and Lemmatization


In [ ]:
# Analyze target variable distribution
print("\n" + "="*70)
print("TARGET VARIABLE DISTRIBUTION")
print("="*70)

target_counts = df['cyberbullying_type'].value_counts()
print(f"\nValue counts:\n{target_counts}")

target_pct = df['cyberbullying_type'].value_counts(normalize=True) * 100
print(f"\nPercentages:\n{target_pct.round(2)}")

# Create visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Bar plot
target_counts.plot(kind='bar', ax=ax1, color='steelblue', edgecolor='black')
ax1.set_title('Distribution of Cyberbullying Types', fontsize=14, fontweight='bold')
ax1.set_xlabel('Cyberbullying Type')
ax1.set_ylabel('Count')
ax1.tick_params(axis='x', rotation=45)
ax1.grid(axis='y', alpha=0.3)

# Pie chart
colors = plt.cm.Set3(np.linspace(0, 1, len(target_counts)))
ax2.pie(target_counts.values, labels=target_counts.index, autopct='%1.1f%%', 
        colors=colors, startangle=90)
ax2.set_title('Proportion of Each Category', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

print("\n✓ Dataset is well-balanced across categories!")


In [ ]:
# Check for missing values
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nTotal missing values: {df.isnull().sum().sum()}")

# Display dataset info
df.info()


In [ ]:
# Load the dataset
df = pd.read_csv('../cyberbullying_tweets.csv')

print("="*70)
print("DATASET OVERVIEW")
print("="*70)
print(f"\nDataset shape: {df.shape[0]} rows, {df.shape[1]} columns")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst 5 rows:")
print(df.head())
print(f"\nData types:\n{df.dtypes}")
print(f"\nBasic statistics:\n{df.describe()}")


## 1. Load and Explore the Dataset

The dataset contains tweets labeled with cyberbullying types. We'll start by loading and analyzing the data to understand its structure and distribution.


In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)
import warnings
warnings.filterwarnings('ignore')

# Configure visualization
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("✓ All libraries imported successfully!")


# Cyberbullying Detection in Tweets Using Machine Learning

**Proyecto de Inteligencia Artificial 2026**
- Opción B: Detección de Ciberacoso en Redes Sociales
- Objetivo: Construir un sistema de clasificación automática para identificar ciberacoso en tweets
- Dataset: 47,000+ tweets etiquetados con categorías de ciberacoso

## Sistema de clasificación multicategoría:
- Age (acoso por edad)
- Ethnicity (acoso por etnicidad)
- Gender (acoso por género)
- Religion (acoso religioso)
- Other Cyberbullying (otro tipo de ciberacoso)
- Not Cyberbullying (no es ciberacoso)

### Fases del Proyecto:
1. Análisis Exploratorio y Limpieza de Datos
2. Pre-procesamiento de Texto (NLP Pipeline)
3. Vectorización (BoW y TF-IDF)
4. División Train-Test
5. Comparación de 5 modelos diferentes
6. Evaluación y Análisis de Resultados
